In [ ]:
# ============================================================================
# STEP 1: INSTALL REQUIRED PACKAGES
# ============================================================================
print("="*70)
print(" INSTALLING PACKAGES")
print("="*70)
print("This will take 2-3 minutes. Please wait...")
print("(Dependency warnings are normal in Colab and can be ignored)\n")

!pip install -q langchain langchain-community langchain-google-genai sentence-transformers chromadb google-generativeai google-search-results pandas 2>&1 | grep -v "dependency conflicts\|incompatible\|ERROR: pip's dependency" || true

print("\n Installation complete!")
print("="*70)

In [ ]:
# ============================================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================================
print("\n Importing libraries...")

import getpass
import os
import pandas as pd
from datetime import datetime

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from langchain.schema import Document
from langchain_community.utilities import SerpAPIWrapper
from langchain.agents import Tool

print(" All libraries imported successfully!")


In [ ]:
# Step  Set up API Key
import getpass
import os
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY") or userdata.get("GEM_API_KEY")
if not api_key:
    raise RuntimeError("Add your Gemini key to Colab userdata as GEMINI_API_KEY (or GEM_API_KEY).")
os.environ["GOOGLE_API_KEY"] = api_key
print("Gemini API key configured.")

In [ ]:
print("\n2. SerpAPI Key (Optional - for web search)")
print("   Get free key from: https://serpapi.com/")
print("   Free tier: 100 searches/month")
print("   Press Enter to skip if you don't have one")
serpapi_key = getpass.getpass("   Enter your SerpAPI Key (or press Enter to skip): ")
if serpapi_key:
    os.environ["SERPAPI_API_KEY"] = serpapi_key
    web_search_enabled = True
    print("    SerpAPI configured - Web search enabled")
else:
    web_search_enabled = False
    print("     Web search disabled (no SerpAPI key)")

print("\n API Keys configured!")
print("="*70)

In [ ]:
# ============================================================================
# STEP 4: CREATE MULTI-SOURCE DATA
# ============================================================================
print("\n" + "="*70)
print(" CREATING MULTI-SOURCE DATA")
print("="*70)

# Source 1: Company Documents (Text)
print("\n Creating company documents...")
company_docs = [
    Document(
        page_content="TechCorp Q4 2024 Financial Report: Revenue reached $10.5M, up 25% YoY. Net profit: $2.1M. Key growth drivers: AI products (+40%) and cloud services (+30%).",
        metadata={"source": "company_financials.pdf", "type": "document", "date": "2024-12-31", "department": "Finance"}
    ),
    Document(
        page_content="TechCorp Remote Work Policy 2025: Employees may work remotely up to 4 days/week. Core hours: 10 AM - 3 PM EST. All remote workers must attend weekly team meetings.",
        metadata={"source": "hr_policies.pdf", "type": "document", "date": "2025-01-15", "department": "HR"}
    ),
    Document(
        page_content="Project Phoenix Status: Phase 2 completed on schedule. Budget: $500K (95% utilized). Team: 12 members. Next milestone: Phase 3 launch in Q2 2025.",
        metadata={"source": "project_status.docx", "type": "document", "date": "2025-01-20", "department": "Engineering"}
    ),
]
print(f"    Created {len(company_docs)} company documents")

# Source 2: Product Catalog (Structured Data)
print("\n Creating product catalog...")
products_data = {
    "Product": ["AI Assistant Pro", "Cloud Storage Plus", "Analytics Dashboard", "Security Suite"],
    "Price": ["$99/month", "$49/month", "$199/month", "$149/month"],
    "Category": ["AI/ML", "Storage", "Analytics", "Security"],
    "Rating": [4.8, 4.5, 4.7, 4.9],
    "Users": ["50K+", "100K+", "25K+", "40K+"]
}
products_df = pd.DataFrame(products_data)

# Convert structured data to documents
product_docs = []
for idx, row in products_df.iterrows():
    content = f"{row['Product']}: ${row['Price']} - {row['Category']} category. Rating: {row['Rating']}/5.0 with {row['Users']} active users."
    product_docs.append(
        Document(
            page_content=content,
            metadata={"source": "product_catalog", "type": "structured_data", "category": row['Category']}
        )
    )
print(f"    Created {len(product_docs)} product entries")

# Source 3: Customer Reviews (Text)
print("\n Creating customer reviews...")
review_docs = [
    Document(
        page_content="Review by John D.: AI Assistant Pro is amazing! Increased our productivity by 40%. Worth every penny. 5 stars!",
        metadata={"source": "customer_reviews", "type": "review", "rating": 5, "product": "AI Assistant Pro"}
    ),
    Document(
        page_content="Review by Sarah M.: Cloud Storage Plus is reliable and fast. Great value for money. Seamless integration with our tools. 4 stars.",
        metadata={"source": "customer_reviews", "type": "review", "rating": 4, "product": "Cloud Storage Plus"}
    ),
    Document(
        page_content="Review by Mike R.: Analytics Dashboard provides incredible insights. The visualizations are top-notch. Highly recommended! 5 stars.",
        metadata={"source": "customer_reviews", "type": "review", "rating": 5, "product": "Analytics Dashboard"}
    ),
]
print(f"   Created {len(review_docs)} customer reviews")

# Source 4: Knowledge Base Articles
print("\n Creating knowledge base articles...")
kb_docs = [
    Document(
        page_content="How to integrate AI Assistant Pro: Use our REST API with your API key. Documentation: docs.techcorp.com/api. Support available 24/7.",
        metadata={"source": "knowledge_base", "type": "kb_article", "topic": "integration"}
    ),
    Document(
        page_content="Security best practices: Enable 2FA, use strong passwords, rotate API keys quarterly, monitor access logs, and enable audit trails.",
        metadata={"source": "knowledge_base", "type": "kb_article", "topic": "security"}
    ),
]
print(f"    Created {len(kb_docs)} knowledge base articles")

# Combine all documents
all_docs = company_docs + product_docs + review_docs + kb_docs
print(f"\n Total documents from all sources: {len(all_docs)}")
print("="*70)


In [ ]:
# ============================================================================
# STEP 5: PROCESS AND INDEX ALL SOURCES
# ============================================================================
print("\n" + "="*70)
print(" PROCESSING MULTI-SOURCE DATA")
print("="*70)

print("\n Chunking documents...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
splits = text_splitter.split_documents(all_docs)
print(f"    Created {len(splits)} chunks")

print("\n Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("   Embedding model loaded")

print("\n Creating vector database...")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="multi_source_rag"
)
print("    Vector database created")

print("\n Setting up retriever...")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("    Retriever configured")

print("\n All sources indexed and ready!")
print("="*70)

In [ ]:
# ============================================================================
# STEP 6: INITIALIZE LLM AND MEMORY
# ============================================================================
print("\n" + "="*70)
print(" INITIALIZING AI COMPONENTS")
print("="*70)

print("\n Initializing Gemini...")
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-exp",
    temperature=0.7,
    max_output_tokens=1024,
)
print("    Gemini ready")

print("\n Setting up conversation memory...")
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)
print("    Memory initialized")

# Optional: Set up web search
if web_search_enabled:
    print("\n Setting up web search...")
    from langchain_community.utilities import SerpAPIWrapper

    # Initialize SerpAPIWrapper with proper return type
    search = SerpAPIWrapper()

    # Create a wrapper function that returns a string
    def search_web_string(query):
        """Wrapper to ensure string output from SerpAPI"""
        try:
            results = search.results(query)  # Use .results() instead of .run()

            if isinstance(results, dict):
                # Extract organic search results
                organic = results.get('organic_results', [])
                snippets = []

                for i, item in enumerate(organic[:3], 1):  # Top 3 results
                    title = item.get('title', 'No title')
                    snippet = item.get('snippet', 'No description')
                    link = item.get('link', '')

                    snippets.append(f"Result {i}:\nTitle: {title}\nDescription: {snippet}\nSource: {link}")

                return "\n\n".join(snippets) if snippets else "No results found"
            else:
                return str(results)
        except Exception as e:
            return f"Web search error: {str(e)}"

    web_search_tool = Tool(
        name="Web Search",
        func=search_web_string,
        description="Search the web for current information"
    )
    print("    Web search ready")
else:
    print("\n Web search disabled (no SerpAPI key)")
    web_search_tool = None

print("\n All AI components ready!")
print("="*70)



In [ ]:
# ============================================================================
# STEP 7: CREATE HYBRID SEARCH FUNCTION
# ============================================================================
print("\n" + "="*70)
print(" BUILDING HYBRID SEARCH SYSTEM")
print("="*70)

class HybridRAG:
    """Combines internal RAG with web search"""

    def __init__(self, vectorstore, llm, memory, web_search_enabled=False, web_search_tool=None):
        self.vectorstore = vectorstore
        self.llm = llm
        self.memory = memory
        self.web_search_enabled = web_search_enabled
        self.web_search_tool = web_search_tool

    def needs_web_search(self, question):
        """Determine if question needs web search"""
        web_keywords = [
            'latest', 'current', 'recent', 'today', 'now', 'news',
            'price', 'weather', 'what is happening', 'developments',
            'trends', 'updates', 'breaking', '2025'
        ]
        question_lower = question.lower()
        return any(keyword in question_lower for keyword in web_keywords)

    def search_internal(self, question):
        """Search internal documents"""
        docs = self.vectorstore.similarity_search(question, k=4)
        if docs:
            context = "\n\n".join([
                f"[{doc.metadata.get('type', 'document')} - {doc.metadata.get('source', 'Unknown')}]\n{doc.page_content}"
                for doc in docs
            ])
            return context, docs
        return "", []

    def search_web(self, question):
        """Search the web"""
        if not self.web_search_enabled or not self.web_search_tool:
            return "", []

        try:
            print("    Searching the web...")
            # The tool now returns a properly formatted string
            result = self.web_search_tool.run(question)

            # Create document with the string content
            web_doc = Document(
                page_content=result,
                metadata={"source": "Web Search", "type": "web"}
            )
            return f"[Web Search Results]\n{result}", [web_doc]

        except Exception as e:
            print(f"     Web search error: {str(e)}")
            return "", []

    def answer_question(self, question):
        """Answer using hybrid approach"""
        print(f"\n Analyzing query...")

        # Check if we need web search
        needs_web = self.needs_web_search(question)

        # Always search internal first
        print("    Searching internal sources...")
        internal_context, internal_docs = self.search_internal(question)

        contexts = []
        all_docs = internal_docs.copy()

        if internal_context:
            contexts.append(internal_context)

        # Add web search if needed
        if needs_web and self.web_search_enabled:
            web_context, web_docs = self.search_web(question)
            if web_context:
                contexts.append(web_context)
                all_docs.extend(web_docs)

        # Combine all contexts
        full_context = "\n\n---\n\n".join(contexts) if contexts else "No relevant information found."

        # Get chat history
        chat_history_str = ""
        if hasattr(self.memory, 'chat_memory'):
            messages = self.memory.chat_memory.messages
            if messages:
                history_parts = []
                for msg in messages[-6:]:  # Last 3 exchanges
                    role = "Human" if msg.type == "human" else "Assistant"
                    history_parts.append(f"{role}: {msg.content[:200]}")
                chat_history_str = "\n".join(history_parts)

        # Generate answer
        print("    Generating answer...")

        prompt = f"""You are a helpful AI assistant with access to multiple data sources:
- Internal documents (company info, products, reviews, knowledge base)
- Web search (for current/latest information)

Previous conversation:
{chat_history_str}

Context from sources:
{full_context}

Question: {question}

Instructions:
- Use the provided context to answer comprehensively
- Cite your sources (e.g., "According to the financial report..." or "Web search shows...")
- If information is from web search, mention it's current/latest information
- If you don't have enough information, say so clearly

Answer:"""

        answer = self.llm.invoke(prompt).content

        # Store in memory
        from langchain.schema import HumanMessage, AIMessage
        self.memory.chat_memory.add_message(HumanMessage(content=question))
        self.memory.chat_memory.add_message(AIMessage(content=answer))

        return {
            "answer": answer,
            "source_documents": all_docs,
            "used_web_search": needs_web and self.web_search_enabled
        }

# Initialize hybrid system
print("\n Creating hybrid RAG system...")
hybrid_rag = HybridRAG(
    vectorstore=vectorstore,
    llm=llm,
    memory=memory,
    web_search_enabled=web_search_enabled,
    web_search_tool=web_search_tool if web_search_enabled else None
)
print("   Hybrid RAG system ready")

print("\n System fully operational!")
print("="*70)


In [ ]:
# ============================================================================
# STEP 8: HELPER FUNCTIONS
# ============================================================================
def analyze_sources(source_docs):
    """Analyze which sources were used"""
    sources_by_type = {}
    for doc in source_docs:
        source_type = doc.metadata.get('type', 'unknown')
        source_name = doc.metadata.get('source', 'unknown')

        if source_type not in sources_by_type:
            sources_by_type[source_type] = set()
        sources_by_type[source_type].add(source_name)

    return sources_by_type

def chat(question, show_sources=True):
    """Chat with hybrid multi-source RAG system"""
    print(f"\n{'='*70}")
    print(f" Question: {question}")
    print(f"{'='*70}")

    result = hybrid_rag.answer_question(question)

    print(f"\n Answer:\n{result['answer']}")

    if show_sources and result['source_documents']:
        print(f"\n Information Sources:")
        print("-"*70)

        sources_analysis = analyze_sources(result['source_documents'])

        for source_type, sources in sources_analysis.items():
            icon = {
                'document': '📄',
                'structured_data': '📊',
                'review': '⭐',
                'kb_article': '📖',
                'web': '🌐'
            }.get(source_type, '📌')

            print(f"{icon} {source_type.replace('_', ' ').title()}:")
            for source in sources:
                print(f"   • {source}")

        if result.get('used_web_search'):
            print("\n This answer includes information from web search")

    print(f"\n{'='*70}\n")
    return result


In [ ]:
# ============================================================================
# STEP 9: DEMO QUERIES
# ============================================================================
print("\n" + "="*70)
print(" MULTI-SOURCE RAG DEMONSTRATION")
print("="*70)
print("\nRunning demo queries that combine multiple data sources...\n")

# Query 1: Financial + Product data
chat("What was our Q4 2024 revenue and what are our top-rated products?")

# Query 2: Product + Reviews
chat("Tell me about the AI Assistant Pro - pricing, rating, and what customers say about it.")

# Query 3: Policy + Knowledge Base
chat("What's our remote work policy and what security practices should remote workers follow?")

# Query 4: Cross-source analysis
chat("Which products have the best customer ratings and how do they contribute to our revenue?")

# Query 5: Project + Product
chat("What's the status of Project Phoenix and which products could benefit from it?")


In [ ]:
# ============================================================================
# STEP 10: INTERACTIVE MODE
# ============================================================================
print("\n" + "="*70)
print(" INTERACTIVE MODE")
print("="*70)
print("\nNow you can ask your own questions!")
print("The system will intelligently search across all sources.")
print("\nCommands:")
print("  'sources' - Show all available sources")
print("  'stats'   - Show source statistics")
print("  'clear'   - Clear conversation memory")
print("  'quit'    - Exit")
print("\n" + "-"*70 + "\n")

while True:
    try:
        user_input = input(" You: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ['quit', 'exit', 'q']:
            print("\n Thanks for trying Multi-Source RAG!")
            break

        if user_input.lower() == 'sources':
            print("\n Available Sources:")
            print("-"*70)
            print(" Documents: company_financials.pdf, hr_policies.pdf, project_status.docx")
            print(" Structured Data: product_catalog")
            print(" Reviews: customer_reviews")
            print(" Knowledge Base: kb_article")
            if web_search_enabled:
                print(" Web Search: Enabled")
            print("-"*70 + "\n")
            continue

        if user_input.lower() == 'stats':
            print(f"\n Data Statistics:")
            print("-"*70)
            print(f"Total documents: {len(all_docs)}")
            print(f"Total chunks: {len(splits)}")
            print(f"Company docs: {len(company_docs)}")
            print(f"Product entries: {len(product_docs)}")
            print(f"Customer reviews: {len(review_docs)}")
            print(f"KB articles: {len(kb_docs)}")
            print("-"*70 + "\n")
            continue

        if user_input.lower() == 'clear':
            memory.clear()
            print("🧹 Memory cleared!\n")
            continue

        # Process question
        chat(user_input)

    except KeyboardInterrupt:
        print("\n\n Chat interrupted. Goodbye!")
        break
    except Exception as e:
        print(f"\n Error: {str(e)}")
        print("Please try again.\n")
